# 04 — Recourse Dataset + Level 1: Prediction Fairness

**Author:** Matteo (fairness / FACTS analysis)

This notebook does two things:
1. Runs DiCE on **every employee the model predicts as at-risk** and saves a
   tidy recourse dataset (this is the input for the FACTS notebook).
2. **Level 1 — Prediction Fairness:** does the model *predict* attrition
   equally across protected groups? (statistical parity, equal opportunity).

It reuses the team's DiCE setup through `utils.py` — no code is duplicated.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dice_ml

import utils  # our shared module (reuses Kamila's DiCE functions)
from utils import (
    ScaledModelWrapper, load_artifacts, load_raw_with_split,
    build_recourse_dataset, PROTECTED_ATTRIBUTES, SEED_DICE,
)

# ── EDIT THESE PATHS to match the repo layout ────────────────────────────────
BASE_PATH = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
RF_DIR    = os.path.join(BASE_PATH, "data", "models", "RF")
XGB_DIR   = os.path.join(BASE_PATH, "data", "models", "XGBoost")
OUT_DIR   = os.path.join(BASE_PATH, "data", "fairness")
os.makedirs(OUT_DIR, exist_ok=True)

THRESHOLD = 0.20   # same decision threshold the models use
N_CF      = 3      # same as the DiCE notebook

ModuleNotFoundError: No module named 'dice_ml'

## 1. Load data (readable demographics) and model artifacts

In [1]:
df_raw, df_encoded, feature_cols = load_raw_with_split()
print(f"Dataset: {df_raw.shape[0]} employees, {len(feature_cols)} features")
print(f"True attrition rate: {df_raw['Attrition'].mean():.1%}")

rf  = load_artifacts(RF_DIR)
xgb = load_artifacts(XGB_DIR)
feature_names = rf["feature_names"]

rf_wrapper  = ScaledModelWrapper(rf["model"],  rf["scaler"],  feature_names)
xgb_wrapper = ScaledModelWrapper(xgb["model"], xgb["scaler"], feature_names)
print("Models + wrappers ready.")

NameError: name 'load_raw_with_split' is not defined

## 2. Build DiCE explainers (reusing the team's configuration)

In [ ]:
# DiCE needs a training dataframe; we use the full encoded frame (unscaled).
dice_df = df_encoded[feature_names].astype("float64").copy()
dice_df["Attrition"] = df_encoded["Attrition"].values

dice_data = dice_ml.Data(
    dataframe=dice_df,
    continuous_features=list(feature_names),
    outcome_name="Attrition",
)
dice_rf  = dice_ml.Model(model=rf_wrapper,  backend="sklearn", model_type="classifier")
dice_xgb = dice_ml.Model(model=xgb_wrapper, backend="sklearn", model_type="classifier")
exp_rf  = dice_ml.Dice(dice_data, dice_rf,  method="random")
exp_xgb = dice_ml.Dice(dice_data, dice_xgb, method="random")
print("DiCE explainers ready.")

## 3. Generate the recourse dataset on ALL at-risk employees

⚠️ This is the slow step. Checkpoints are written every 25 employees so a crash
never loses progress. Some employees will have **no valid counterfactual** —
that is itself a fairness signal (effectiveness), not an error.

In [ ]:
recourse_rf = build_recourse_dataset(
    exp_rf, rf_wrapper, df_encoded, df_raw, feature_names,
    le_dict=rf["label_encoders"], n_cf=N_CF, seed=SEED_DICE,
    threshold=THRESHOLD,
    checkpoint_path=os.path.join(OUT_DIR, "recourse_RF.csv"),
)

In [ ]:
recourse_xgb = build_recourse_dataset(
    exp_xgb, xgb_wrapper, df_encoded, df_raw, feature_names,
    le_dict=xgb["label_encoders"], n_cf=N_CF, seed=SEED_DICE,
    threshold=THRESHOLD,
    checkpoint_path=os.path.join(OUT_DIR, "recourse_XGB.csv"),
)

print("\nRecourse datasets saved. Preview (RF):")
recourse_rf.head()

## 4. LEVEL 1 — Prediction Fairness

Three classic group-fairness notions from the lecture slides:
- **Statistical / Demographic parity:** same predicted-positive rate per group.
- **Equal opportunity:** same true-positive rate (recall) per group.
- **Predictive equality:** same false-positive rate per group.

We compute them for each model and each protected attribute.

In [ ]:
def prediction_fairness(wrapper, df_encoded, df_raw, feature_names,
                        protected_attr, threshold=THRESHOLD):
    """Return a per-group table of prediction-fairness metrics."""
    X = df_encoded[feature_names].astype("float64")
    proba = wrapper.predict_proba(X)[:, 1]
    y_pred = (proba >= threshold).astype(int)
    y_true = df_raw["Attrition"].values

    rows = []
    for g in sorted(df_raw[protected_attr].unique()):
        mask = (df_raw[protected_attr] == g).values
        yt, yp = y_true[mask], y_pred[mask]
        pos = yt == 1
        neg = yt == 0
        rows.append({
            protected_attr:        g,
            "n":                   int(mask.sum()),
            "pred_positive_rate":  round(yp.mean(), 3),                       # statistical parity
            "TPR_recall":          round(yp[pos].mean(), 3) if pos.any() else np.nan,  # equal opportunity
            "FPR":                 round(yp[neg].mean(), 3) if neg.any() else np.nan,  # predictive equality
        })
    return pd.DataFrame(rows)


def fairness_gaps(table, attr):
    """Max-min gap across groups for each metric (0 = perfectly fair)."""
    metrics = ["pred_positive_rate", "TPR_recall", "FPR"]
    return {m: round(table[m].max() - table[m].min(), 3) for m in metrics}

In [ ]:
for model_name, wrapper in [("Random Forest", rf_wrapper), ("XGBoost", xgb_wrapper)]:
    print(f"\n{'='*60}\n{model_name} — Prediction Fairness\n{'='*60}")
    for attr in PROTECTED_ATTRIBUTES:
        tbl = prediction_fairness(wrapper, df_encoded, df_raw, feature_names, attr)
        gaps = fairness_gaps(tbl, attr)
        print(f"\n-- {attr} --")
        print(tbl.to_string(index=False))
        print(f"   gaps (max-min): {gaps}")

## 5. Visualise prediction-positive rates by group

In [ ]:
fig, axes = plt.subplots(1, len(PROTECTED_ATTRIBUTES),
                         figsize=(5 * len(PROTECTED_ATTRIBUTES), 4))
for ax, attr in zip(axes, PROTECTED_ATTRIBUTES):
    rf_tbl  = prediction_fairness(rf_wrapper,  df_encoded, df_raw, feature_names, attr)
    xgb_tbl = prediction_fairness(xgb_wrapper, df_encoded, df_raw, feature_names, attr)
    x = np.arange(len(rf_tbl))
    ax.bar(x - 0.2, rf_tbl["pred_positive_rate"],  0.4, label="RF",  color="#2196F3")
    ax.bar(x + 0.2, xgb_tbl["pred_positive_rate"], 0.4, label="XGB", color="#FF9800")
    ax.set_xticks(x); ax.set_xticklabels(rf_tbl[attr], rotation=20, ha="right")
    ax.set_title(f"Predicted attrition rate — {attr}")
    ax.set_ylabel("rate"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "level1_prediction_fairness.png"), dpi=120)
plt.show()

### Reading Level 1
If the gaps are small, the model looks "fair" by prediction. Keep this in mind:
Von Kügelgen (2022) shows that **fair prediction does not imply fair recourse**.
The FACTS notebook tests exactly that.